# Automatic environment test (operational notebook)
Provisioning executes this notebook using the assessment kernel. It checks installed imports, saved-model compatibility and finite inference without training or accessing holdout data. No assessment conclusions are generated.


In [ ]:
from pathlib import Path
import os, sys, json, importlib.metadata as metadata
os.environ["DO_NOT_TRACK"] = "1"
cwd = Path.cwd().resolve()
ROOT = next(p for p in [cwd, cwd/"support"] if (p/"labkit.py").exists())
sys.path.insert(0, str(ROOT))
assert Path(sys.executable).parent.parent.name == "e179-env", "Wrong environment"
import numpy as np, pandas as pd, scipy, sklearn, xgboost, shap, lime, evidently
import matplotlib, seaborn, joblib, nbformat, nbclient, ipykernel, boto3, docx
from labkit import FEATURES, load_models, load
models = load_models()
demo = load("demo_sample")
rows = []
assert len(FEATURES) == 30
for name, model in models.items():
    scores = model.predict_proba(demo[FEATURES])[:,1]
    assert scores.shape == (len(demo),)
    assert np.isfinite(scores).all() and ((scores>=0)&(scores<=1)).all()
    rows.append({"candidate": name, "rows_scored": len(scores), "features": len(FEATURES)})
packages=["numpy","pandas","scipy","scikit-learn","xgboost-cpu","shap","lime","evidently","ipykernel"]
result={"passed": True, "python": sys.version.split()[0], "models": rows, "packages": {p:metadata.version(p) for p in packages}}
temp = ROOT/".environment-results.tmp"
temp.write_text(json.dumps(result,indent=2))
temp.replace(ROOT/".environment-results.json")
print("AUTOMATIC MODEL AND ENVIRONMENT TEST PASSED")
